# Improved Speech Emotion Recognition (SER) with CNN-LSTM

This notebook implements significant improvements over the previous model:
1. **Temporal Features:** Instead of averaging MFCCs, we keep the time sequence (Time Steps x Features).
2. **Data Augmentation:** Adds noise, pitch shifting, and time stretching to make the model robust.
3. **Hybrid Architecture:** Uses 1D CNNs to extract local features and LSTM layers to capture long-term dependencies.
4. **Regularization:** Includes Dropout and Early Stopping to prevent overfitting.

In [ ]:
import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Flatten, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
# --- CONFIGURATION ---
DATA_PATH = "ravdess"
SAMPLE_RATE = 22050
DURATION = 3.0  # Slightly longer to capture full utterances
N_MFCC = 40
# Calculate expected number of frames based on duration & sample rate
# default hop_length is 512 in librosa
EXPECTED_FRAMES = int(np.ceil((SAMPLE_RATE * DURATION) / 512))
print(f"Target Frame Length: {EXPECTED_FRAMES}")

In [ ]:
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

## 1. Data Augmentation & Feature Extraction Functions

In [ ]:
def noise(data, noise_factor=0.005):
    noise = np.random.randn(len(data)) 
    augmented_data = data + noise_factor * noise
    # Cast back to float32
    augmented_data = augmented_data.astype(type(data[0]))
    return augmented_data

def pitch(data, sampling_rate, pitch_factor=0.7):
    return librosa.effects.pitch_shift(data, sr=sampling_rate, n_steps=pitch_factor)

def stretch(data, rate=0.8):
    return librosa.effects.time_stretch(data, rate=rate)

def extract_features(data, sample_rate):
    # Extract MFCCs without averaging over time
    # Transpose result to shape (Time, Features)
    mfccs = librosa.feature.mfcc(y=data, sr=sample_rate, n_mfcc=N_MFCC)
    return mfccs.T

## 2. Load and Process Dataset

In [ ]:
X, y = [], []

print("Starting processing...")

for actor_dir in tqdm(os.listdir(DATA_PATH)):
    if 'Actor_' in actor_dir:
        actor_path = os.path.join(DATA_PATH, actor_dir)
        for file_name in os.listdir(actor_path):
            if file_name.endswith(".wav"):
                file_path = os.path.join(actor_path, file_name)
                
                # Get Emotion Label
                part = file_name.split('-')
                emotion = emotion_map[part[2]]
                
                # Load Audio
                # Load with fixed duration to ensure consistency, but allow padding later
                data, _ = librosa.load(file_path, duration=DURATION, offset=0.5, sr=SAMPLE_RATE)
                
                # Pad or Truncate to ensure fixed length
                target_len = int(DURATION * SAMPLE_RATE)
                if len(data) < target_len:
                    data = np.pad(data, (0, target_len - len(data)), 'constant')
                else:
                    data = data[:target_len]

                # 1. Original Data
                res1 = extract_features(data, SAMPLE_RATE)
                X.append(res1)
                y.append(emotion)
                
                # 2. Data with Noise
                noise_data = noise(data)
                res2 = extract_features(noise_data, SAMPLE_RATE)
                X.append(res2)
                y.append(emotion)
                
                # 3. Data with Pitch Shift
                pitch_data = pitch(data, SAMPLE_RATE)
                res3 = extract_features(pitch_data, SAMPLE_RATE)
                X.append(res3)
                y.append(emotion)
                
                # 4. Data with Stretch (Speed Change)
                # Note: Stretch changes length, so we must re-pad/truncate
                stretch_data = stretch(data)
                if len(stretch_data) < target_len:
                    stretch_data = np.pad(stretch_data, (0, target_len - len(stretch_data)), 'constant')
                else:
                    stretch_data = stretch_data[:target_len]
                    
                res4 = extract_features(stretch_data, SAMPLE_RATE)
                X.append(res4)
                y.append(emotion)

In [ ]:
# Convert to Numpy Arrays
X = np.array(X)
y = np.array(y)

# Verify Shapes
# Expecting (Samples, TimeSteps, Features)
print(f"X Shape: {X.shape}")
print(f"y Shape: {y.shape}")

# Encode Labels
lb = LabelEncoder()
y = to_categorical(lb.fit_transform(y))
print(f"Classes: {lb.classes_}")
np.save('emotion_classes.npy', lb.classes_)

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 3. Build CNN-LSTM Model

In [ ]:
model = Sequential([
    Input(shape=(X.shape[1], X.shape[2])),
    
    # CNN Block 1 - Extract local patterns
    Conv1D(256, kernel_size=5, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=5, padding='same'),
    Dropout(0.2),
    
    # CNN Block 2
    Conv1D(128, kernel_size=5, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=5, padding='same'),
    Dropout(0.2),
    
    # CNN Block 3
    Conv1D(64, kernel_size=5, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=5, padding='same'),
    Dropout(0.2),
    
    # LSTM Block - Capture sequence over time
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    
    # Classification
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(8, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train with Callbacks

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, verbose=1)

history = model.fit(
    X_train, y_train, 
    epochs=70, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")
model.save('my_audio_emotion_model.h5')